# Trumen — official run on a T4 (Google Colab)

Colab's T4 is the same GPU class as the organisers' machine, so this notebook
1. clones the repository and installs the pinned requirements,
2. downloads the four sample videos from the organisers' Google Drive links,
3. runs the **unchanged** harness exactly as the judges will,
4. checks the time budget (≤ 3× video length) and writes `predictions_samples.json`.

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

In [ ]:
REPO = "https://github.com/trumen-com/road-vision.git"   # <- set this
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!git clone -q $REPO trumen && cd trumen && git log -1 --oneline

In [ ]:
%cd /content/trumen
!pip -q install -r requirements.txt gdown
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# the four sample videos (organisers' Google Drive links); gdown keeps the original file names
IDS = ["1kR9jODA2Wotw4gwkvpRKdqFADNJNc1nS", "1hp8DYeqtYHSwfM6qAo9FPSRHlpMFrIN_",
       "10cHEReCWzO3u-Vk1CnNgHAx6egGy5MwJ", "1aJ-QsAZVYJtLKHiRvKKeBq1D3GWNobRd"]
import os, gdown
os.makedirs("/content/samples", exist_ok=True)
for fid in IDS:
    gdown.download(id=fid, output="/content/samples/", quiet=False)
!ls -la /content/samples

In [ ]:
# offline, exactly as the judges run it
!cd /content/trumen && HTTPS_PROXY=http://127.0.0.1:9 HTTP_PROXY=http://127.0.0.1:9 \
    python run_submission.py --videos /content/samples --out predictions_samples.json --team Trumen
!python evaluate.py --pred predictions_samples.json --validate-only

In [ ]:
import json
p = json.load(open("predictions_samples.json"))
for vid, log in p["log"].items():
    ok = log["total_sec"] <= log["budget_sec"]
    print(f"{vid}: {log['total_sec']:.0f}s of {log['budget_sec']:.0f}s budget "
          f"(A {log.get('part_a_sec')}s, B {log.get('part_b_sec')}s) {'OK' if ok else 'OVER BUDGET'}; "
          f"{len(p['videos'][vid]['events'])} events; errors: {log['errors'][:1]}")

In [ ]:
# download the result and commit it to the repository as predictions_samples.json
from google.colab import files
files.download("predictions_samples.json")